<a href="https://colab.research.google.com/github/MiguelCortezPino/ProgramacionV-Marzo29/blob/master/notebook/Siniestros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:

url = "https://raw.githubusercontent.com/MiguelCortezPino/etl-data-pipeline/refs/heads/main/data/raw/siniestros.csv"

df = pd.read_csv(url)

df.head()

,id_siniestro,id_poliza,fecha_siniestro,monto_siniestro,estado
0,1,17400,16-10-2025,2092.59,ABIERTO
1,2,2465,01/08/2025,"7,076.25",Abierto
2,3,15785,19-09-2025,702.27,cerrado
3,4,14299,27/09/2025,274.63,Abierto
4,5,12908,01/12/2025,"9,377.69",Rechazado


In [3]:
#Exploracion de datos
df.shape
df.columns
df.info()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4620 entries, 0 to 4619
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_siniestro     4620 non-null   int64 
 1   id_poliza        4620 non-null   int64 
 2   fecha_siniestro  4620 non-null   object
 3   monto_siniestro  4004 non-null   object
 4   estado           3322 non-null   object
dtypes: int64(2), object(3)
memory usage: 180.6+ KB


,0
id_siniestro,0
id_poliza,0
fecha_siniestro,0
monto_siniestro,616
estado,1298


In [4]:
#Limpiezad de datos
siniestros=df.copy()
print(siniestros['fecha_siniestro'].value_counts(dropna=False))


fecha_siniestro
2026/02/24    9
2025-08-11    9
2025-07-24    8
2025-10-25    8
29/06/2025    7
             ..
04-11-2025    1
2025-06-17    1
15/11/2025    1
2025/10/02    1
23/02/2026    1
Name: count, Length: 1850, dtype: int64


In [5]:
siniestros['fecha_siniestro'] = pd.to_datetime(siniestros['fecha_siniestro'], format='mixed', dayfirst=False, errors='coerce')
print(siniestros['fecha_siniestro'].value_counts(dropna=False))


fecha_siniestro
2025-12-11    20
2025-04-06    20
2025-10-26    20
2025-06-29    19
2025-08-08    19
              ..
2025-09-25     2
2025-09-07     2
2026-03-01     1
2026-10-02     1
2026-08-01     1
Name: count, Length: 441, dtype: int64


In [6]:
#Funcion para estandarizar los numeros
def limpiar_numero_mixto(v):
    if pd.isna(v): return None

    s = str(v).strip().replace('$', '').replace(' ', '')

    if s in ('', '-', 'N/A', 'nan', 'None'): return None

    if ',' in s and '.' in s:
        s = s.replace('.' if s.find('.') < s.find(',') else ',', '')

    try:
        return float(s.replace(',', '.'))
    except:
        return None

In [7]:
siniestros['monto_siniestro'] = siniestros['monto_siniestro'].apply(limpiar_numero_mixto)
print(siniestros['monto_siniestro'].head(10))

0     2092.59
1     7076.25
2      702.27
3      274.63
4     9377.69
5    10535.74
6    10513.30
7     2736.20
8         NaN
9     8801.03
Name: monto_siniestro, dtype: float64


In [8]:
siniestros=df.copy()

print(siniestros['estado'].value_counts(dropna=False))

estado
NaN          1298
Abierto       700
cerrado       677
ABIERTO       664
Cerrado       645
Rechazado     636
Name: count, dtype: int64


In [9]:
siniestros['estado'] = siniestros['estado'].str.strip().str.title()
print(siniestros['estado'].value_counts(dropna=False))

estado
Abierto      1364
Cerrado      1322
NaN          1298
Rechazado     636
Name: count, dtype: int64


In [11]:
#Separar datos válidos y rechazados
filtrado_validos = (siniestros['fecha_siniestro'].notna()) & \
                    (siniestros['monto_siniestro'].notna()) & \
                    (siniestros['estado'].notna())
validos = siniestros[filtrado_validos].copy()
rechazados = siniestros[~filtrado_validos].copy()

In [13]:
#Motivo de rechazo
def motivo_rechazo(row):
  motivos = []
  if pd.isnull(row['fecha_siniestro']):
    motivos.append('fecha_siniestro_vacio')
  if pd.isnull(row['monto_siniestro']):
    motivos.append('monto_siniestro_vacio')
  if pd.isnull(row['estado']):
    motivos.append('estado_vacio')
  return ','.join(motivos)

  rechazados['motivo_rechazo'] = rechazados.apply(motivo_rechazo, axis=1)



In [14]:
validos.to_csv("siniestros_curated.csv", index=False)

rechazados.to_csv("siniestros_rejects.csv", index=False)
